# Dynamics 04c (Rust) — Coopération payorienne : le FairBot qui se prouve par cadres de Kripke

**Navigation** : [<< 04b-FairBot-Loeb](Dynamics-04b-FairBot-Loeb-Lean.ipynb) | [Version Python](Dynamics-04c-Cooperation-Payorienne-Kripke-Python.ipynb) | [Version C#](Dynamics-04c-Cooperation-Payorienne-Kripke-CSharp.ipynb) | [Index](README.md) | [Dynamics-05-Agents-Bornes-Python >>](Dynamics-05-Agents-Bornes-Python.ipynb)

Ce carnet sert la tranche T-2 du plan de consolidation de l'[issue #26985](https://github.com/jsboige/CoursIA/issues/26985), née du Concern de l'issue #26954 : l'état de l'art ouvert de la théorie des jeux (OSGT) n'était cité nulle part dans le dépôt. Il reprend la **coopération payorienne** du post d'Alignment Forum *« Payorian cooperation is easy with Kripke frames »* (transhumanist_atom_understander, 9 mars 2026), lu dans sa copie archivée au gisement bibliographique (`GameTheory/Cooperation, Institutions and LLM Agents/2026-03-09 - transhumanist_atom_understander - Payorian cooperation is easy with Kripke frames - Alignment Forum LaCP6WyNzX8kiZn3w.pdf`).

Le carnet fait trois choses, et une seule de chaque :

1. **énonce** le FairBot payorien `P ↔ □(□P → Q)` et le lemme de Payor, en citant la source primaire — aucune preuve formelle n'est refaite ici ;
2. **implémente** le bot dans l'esprit du moteur jouet de [Dynamics-04](Dynamics-04-Open-Source-Game-Theory-Python.ipynb) ;
3. **mesure** la profondeur de réfutation par cadres de Kripke, avec un moteur déclaré en **copie pédagogique** du laboratoire [Tweety-03b](../../SymbolicAI/Tweety/Tweety-03b-Modal-Lab-Lean-Python.ipynb) §3.

La frontière mesure/preuve est tenue partout : les réfutations sont des **calculs bornés**, pas des théorèmes ; le côté formel vit dans [`ProgramGames.FairBot`](../game_theory_lean/ProgramGames/FairBot.lean) (niveau L2) et dans [Dynamics-04b](Dynamics-04b-FairBot-Loeb-Lean.ipynb) pour la partie löbienne.


## 1. Le tournoi et la question

Le tournoi de MIRI est un dilemme du prisonnier **one-shot** où chaque programme lit le code de l'autre — ou plutôt une description de son comportement en logique de la prouvabilité de Gödel-Löb (GL). FairBot est défini par « je coopère avec toi si tu coopères avec moi, prouvablement » :

$$F(X) \leftrightarrow \Box\, X(F)$$

Malgré la circularité apparente, deux FairBots coopèrent : c'est la **coopération löbienne**, qui passe par le théorème de Löb.

Critch a proposé une autre voie, le **lemme de Payor** : au lieu de vérifier que l'adversaire coopère, on vérifie que *sa propre coopération prouvable implique* celle de l'adversaire :

$$P \leftrightarrow \Box(\Box P \to Q)$$

avec $P \equiv F(X)$ et $Q \equiv X(F)$. La question de ce carnet : **pourquoi cette forme se réfute-t-elle plus facilement sur un cadre de Kripke ?** Le post répond « deux mondes au lieu de quatre » ; nous allons le mesurer.

Les paiements sont ceux du moteur de Dynamics-04.

In [1]:
// --- Paiements du dilemme du prisonnier (meme convention que Dynamics-04) ---
// Les constantes sont au niveau ITEM : une `fn` imbriquee ne capture pas un
// `let` de cellule.
const T: i32 = 5;
const R: i32 = 3;
const P_PAIEMENT: i32 = 1;
const S: i32 = 0;

/// Le paiement du joueur qui joue `mon_action` contre `action_adverse`.
#[allow(dead_code)]
fn payoff_self(mon_action: &str, action_adverse: &str) -> i32 {
    if mon_action == "C" {
        if action_adverse == "C" { R } else { S }
    } else if action_adverse == "C" { T } else { P_PAIEMENT }
}

assert!(T > R && R > P_PAIEMENT && P_PAIEMENT > S, "parametres PD non canoniques");
assert!(2 * R > T + S, "cooperation mutuelle > alternance");

println!("self  other  payoff");
for me in ["C", "D"] {
    for other in ["C", "D"] {
        println!("  {}     {}     {}", me, other, payoff_self(me, other));
    }
}
// La cellule se termine sur une instruction : evcxr n'affiche rien apres.
let _ = ();

self  other  payoff


  C     C     3


  C     D     0


  D     C     5


  D     D     1


### Lecture : la matrice tient la structure du dilemme

La sortie donne exactement quatre paiements — `C/C → 3`, `C/D → 0`, `D/C → 5`, `D/D → 1` — et les deux assertions en tête de cellule fixent l'ordre strict `T > R > P > S` ainsi que la préférence pour la coopération mutuelle sur l'alternance. C'est cette matrice que les bots ci-dessous vont jouer, sans itération ni réputation : tout se décide avant le coup, sur la seule lecture du code adverse.

## 2. Le lemme de Payor, et ce que ce carnet n'en fait pas

**Lemme de Payor.** Pour toute formule $A$ et tout $B$ : $\vdash \Box(\Box A \to B) \to A$.

L'attribution est double, et le post la donne telle quelle : James Payor raconte dans le fil être arrivé à `□(□A→B)→A` en raisonnant sur les étapes de l'algorithme, et c'est **Andrew Critch** qui propose de s'en servir à la place du théorème de Löb pour prouver la coopération de soi — « *Instead of Löb's theorem, we use what he calls "Payor's lemma"* ». Le nom est resté au lemme, l'usage au post.

La preuve est **arithmétique** — elle dérive du théorème de Löb — et n'est pas refaite ici : le carnet travaille exclusivement du côté **sémantique**, où le lemme se lit comme une contrainte sur les cadres. L'axiome de Löb $\Box(\Box A \to A) \to \Box A$ y est valide exactement sur les cadres transitifs et **bien fondés**.

Déclarations de provenance :

- **copie pédagogique** : le moteur de Kripke de la cellule suivante est une copie déclarée du §3 du laboratoire Tweety-03b (mêmes variantes d'un enum récursif, même clause de forcing), motif = un carnet est autonome et ne peut pas importer un autre carnet ;
- **citation** : l'énoncé du lemme et les dérivations du post sont cités, jamais re-démontrés — le côté mécanisé vit dans le lake `game_theory_lean` (niveau L2) et, pour la partie löbienne, dans le carnet 04b ;
- **archive** : la copie du post lue pour ce carnet est celle du gisement bibliographique, pas une page web revisitée.

In [2]:
// --- Moteur de Kripke : AST gelee + forcing recursif ---
// COPIE PEDAGOGIQUE DECLAREE du §3 de Tweety-03b-Modal-Lab-Lean-Python.ipynb :
// meme AST (un enum recursif au lieu des dataclasses gelees) et meme clause de
// forcing. Motif : un carnet est autonome.
use std::collections::{HashMap, HashSet};

/// AST des formules modales : la recursion est portee par `Box`.
/// `Clone + PartialEq + Eq + Hash` reproduit la dataclass `frozen` Python :
/// l'egalite et le hachage sont structurels.
#[derive(Clone, PartialEq, Eq, Hash)]
#[allow(dead_code)]
enum Formule {
    Atome(String),
    Non(Box<Formule>),
    Imp(Box<Formule>, Box<Formule>),
    Et(Box<Formule>, Box<Formule>),
    Boite(Box<Formule>),
    Diamant(Box<Formule>),
}

/// Un atome du langage modal.
#[allow(dead_code)]
fn atome(nom: &str) -> Formule { Formule::Atome(nom.to_string()) }

/// Cadre de Kripke : mondes, relation de visibilite, valuation des atomes.
#[allow(dead_code)]
struct Modele {
    mondes: Vec<usize>,
    relation: HashSet<(usize, usize)>,
    valuation: HashMap<String, HashSet<usize>>,
}

/// Les mondes visibles depuis `x` (le `frozenset` de la source ; l'ordre
/// n'importe qu'aux `all` / `any`).
#[allow(dead_code)]
fn successeurs(m: &Modele, x: usize) -> Vec<usize> {
    m.relation.iter().filter(|(a, _)| *a == x).map(|(_, y)| *y).collect()
}

/// Forcing : `Boite` = tous les successeurs, `Diamant` = au moins un.
#[allow(dead_code)]
fn satisfait(m: &Modele, x: usize, f: &Formule) -> bool {
    match f {
        Formule::Atome(nom) => m.valuation.get(nom).expect("atome du modele").contains(&x),
        Formule::Non(g) => !satisfait(m, x, g),
        Formule::Imp(a, b) => !satisfait(m, x, a) || satisfait(m, x, b),
        Formule::Et(a, b) => satisfait(m, x, a) && satisfait(m, x, b),
        Formule::Boite(g) => successeurs(m, x).iter().all(|y| satisfait(m, *y, g)),
        Formule::Diamant(g) => successeurs(m, x).iter().any(|y| satisfait(m, *y, g)),
    }
}

/// Impresseur unique des formules : jamais `{:?}` (le `repr` Python n'a pas
/// d'equivalent lisible, et `show` est le seul imprimeur du carnet).
#[allow(dead_code)]
fn show(f: &Formule) -> String {
    match f {
        Formule::Atome(nom) => nom.clone(),
        Formule::Non(g) => format!("non {}", show(g)),
        Formule::Imp(a, b) => format!("({} -> {})", show(a), show(b)),
        Formule::Et(a, b) => format!("({} et {})", show(a), show(b)),
        Formule::Boite(g) => format!("[]{}", show(g)),
        Formule::Diamant(g) => format!("<>{}", show(g)),
    }
}

// Mini-test : le monde 0 voit le monde 1 ; P est vrai en 1 seulement.
let m_test = Modele {
    mondes: vec![0, 1],
    relation: HashSet::from([(0usize, 1usize)]),
    valuation: HashMap::from([("P".to_string(), HashSet::from([1usize]))]),
};
// Ordre d'insertion preserve : un `BTreeMap` aurait trie les etiquettes.
let verdicts: Vec<(&str, bool)> = vec![
    ("P en 0", satisfait(&m_test, 0, &atome("P"))),
    ("[]P en 0", satisfait(&m_test, 0, &Formule::Boite(Box::new(atome("P"))))),
    ("[]P -> P en 0", satisfait(&m_test, 0, &Formule::Imp(Box::new(Formule::Boite(Box::new(atome("P")))), Box::new(atome("P"))))),
    ("<>P en 1", satisfait(&m_test, 1, &Formule::Diamant(Box::new(atome("P"))))),
];
for (etiquette, valeur) in &verdicts {
    println!("  {:<14} : {}", etiquette, if *valeur { "vrai" } else { "faux" });
}
assert!(verdicts[1].1 && !verdicts[2].1);
println!("\nMoteur operationnel : au monde mort (1 sans successeur), []P est vacuement vrai.");

  P en 0         : faux


  []P en 0       : vrai


  []P -> P en 0  : faux


  <>P en 1       : faux


Moteur operationnel : au monde mort (1 sans successeur), []P est vacuement vrai.


### Lecture : le monde mort et la vacuité

`[]P en 0` est vrai alors que `[]P -> P en 0` est faux — la sortie montre les deux lignes opposées, et l'assertion finale le confirme. Le monde 1, sans successeur, rend `[]` vacuement vrai ; c'est exactement la clause qui portera la réfutation : une boîte fausse signale toujours un monde *visible* où son contenu échoue.

## 3. Réfuter en empilant des contraintes sur un cadre

La méthode du post est un raisonnement par l'absurde sémantique. Pour prouver que FairBot coopère, on suppose qu'il **défecte** : $\neg P$. Comme $P \leftrightarrow \Box(\Box P \to Q)$, cela donne

$$\neg\Box(\Box P \to Q) \;\equiv\; \Diamond(\Box P \land \neg Q)$$

c'est-à-dire un monde visible $w_1$ où $\Box P$ et $\neg Q$ tiennent. On propage ensuite les boîtes le long de la visibilité — transitivité : si $w_0$ voit $w_1$ et $w_1$ voit $w_2$, alors $w_0$ voit $w_2$ — et l'on cherche une **contradiction locale**, un atome et sa négation au même monde.

La cellule suivante code cet algorithme : un dérouleur qui, à chaque obligation de possibilité non satisfaite, ouvre **un** monde de plus (chaîne croissante), propage les nécessités, et s'arrête sur contradiction. Deux limites sont assumées d'emblée : la borne de profondeur rend un statut INCOMPLET plutôt que « cohérent », et le dérouleur n'encode **pas** la bonne-fondation des cadres de GL — la section 5 montrera précisément ce que cela change pour le côté löbien.

In [3]:
// --- Derouler de refutation : saturer des contraintes jusqu'a contradiction ---
// L'algorithme du post, borne en profondeur. Ce n'est PAS un prouveur de GL :
// au-dela de la borne, une obligation ouverte rend INCOMPLET, jamais "coherent".
use std::collections::{BTreeMap, HashSet};

/// AST des formules modales : la recursion est portee par `Box`.
/// `Clone + PartialEq + Eq + Hash` reproduit la dataclass `frozen` Python :
/// l'egalite et le hachage sont structurels.
#[derive(Clone, PartialEq, Eq, Hash)]
#[allow(dead_code)]
enum Formule {
    Atome(String),
    Non(Box<Formule>),
    Imp(Box<Formule>, Box<Formule>),
    Et(Box<Formule>, Box<Formule>),
    Boite(Box<Formule>),
    Diamant(Box<Formule>),
}

/// Un atome du langage modal.
#[allow(dead_code)]
fn a(nom: &str) -> Formule { Formule::Atome(nom.to_string()) }
/// La negation d'une formule.
#[allow(dead_code)]
fn non(f: Formule) -> Formule { Formule::Non(Box::new(f)) }
/// L'implication.
#[allow(dead_code)]
fn imp(a: Formule, b: Formule) -> Formule { Formule::Imp(Box::new(a), Box::new(b)) }
/// La boite (necessite) : `Box` est un mot-cle, d'ou le nom `box_`.
#[allow(dead_code)]
fn box_(f: Formule) -> Formule { Formule::Boite(Box::new(f)) }
/// Le diamant (possibilite).
#[allow(dead_code)]
fn dia(f: Formule) -> Formule { Formule::Diamant(Box::new(f)) }
/// L'equivalence : `f <-> g`.
#[allow(dead_code)]
fn equiv(f: Formule, g: Formule) -> Formule {
    Formule::Et(Box::new(imp(f.clone(), g.clone())), Box::new(imp(g, f)))
}

/// Impresseur unique des formules : jamais `{:?}` (le `repr` Python n'a pas
/// d'equivalent lisible, et `show` est le seul imprimeur du carnet).
#[allow(dead_code)]
fn show(f: &Formule) -> String {
    match f {
        Formule::Atome(nom) => nom.clone(),
        Formule::Non(g) => format!("non {}", show(g)),
        Formule::Imp(a, b) => format!("({} -> {})", show(a), show(b)),
        Formule::Et(a, b) => format!("({} et {})", show(a), show(b)),
        Formule::Boite(g) => format!("[]{}", show(g)),
        Formule::Diamant(g) => format!("<>{}", show(g)),
    }
}

/// L'exception Python devient une valeur : le rolleur la propage par `Result`.
#[allow(dead_code)]
struct Contradiction { monde: usize, f: Formule }

/// Chaine transitive : le monde i voit tout monde cree apres lui.
#[allow(dead_code)]
fn voit(i: usize, j: usize) -> bool { i < j }

/// La formule est-elle deja presente au monde `w` ?
#[allow(dead_code)]
fn present(mondes: &BTreeMap<usize, HashSet<Formule>>, w: usize, f: &Formule) -> bool {
    mondes.get(&w).expect("monde present").contains(f)
}

/// Ajout d'une formule au monde `w`, avec detection de contradiction.
///
/// Asymetrie mesuree : si `non(f)` est deja present, l'erreur porte `f` ;
/// si `f` est `non(g)` et `g` est present, l'erreur porte `g` (la formule
/// interne, positive). Inverser les deux branches changerait les deux
/// messages affiches par la table.
#[allow(dead_code)]
fn ajoute(mondes: &mut BTreeMap<usize, HashSet<Formule>>, w: usize, f: Formule) -> Result<(), Contradiction> {
    if present(mondes, w, &Formule::Non(Box::new(f.clone()))) {
        return Err(Contradiction { monde: w, f: f.clone() });
    }
    if let Formule::Non(g) = &f {
        if present(mondes, w, g) {
            return Err(Contradiction { monde: w, f: (**g).clone() });
        }
    }
    mondes.get_mut(&w).expect("monde present").insert(f);
    Ok(())
}

/// Fin de la saturation : modele fini, ou borne atteinte.
#[allow(dead_code)]
enum Fin { Sat, Incomplet }

/// Verdict du rouleau : contradiction, modele fini, ou borne atteinte.
#[allow(dead_code)]
enum Resultat {
    Unsat(Contradiction),
    Sat(BTreeMap<usize, HashSet<Formule>>),
    Incomplet,
}

/// Saturation : on decompose les formules jusqu'a ce qu'aucune passe ne change rien.
///
/// L'ordre des bras suit celui des `if/elif` de la source : `non (non f)`,
/// `et`, `imp`, `non (a -> b)`, `box`, `non (diamant)`, `non (box)`,
/// `diamant`. `non (a et b)` ne tombe dans aucun bras : c'est la limite
/// assumee du carnet.
#[allow(dead_code)]
fn saturer(mondes: &mut BTreeMap<usize, HashSet<Formule>>, max_profondeur: usize) -> Result<Fin, Contradiction> {
    loop {
        let mut change = false;
        // Instantanés : Python itere sur des copies (`sorted(list(mondes))`),
        // et mute le dict pendant la passe.
        let cles: Vec<usize> = mondes.keys().cloned().collect();
        for w in cles {
            let mut fs: Vec<Formule> = mondes.get(&w).expect("monde present").iter().cloned().collect();
            fs.sort_by_key(|f| show(f));
            for f in fs {
                match &f {
                    Formule::Non(inner) => {
                        if let Formule::Non(inner2) = &**inner {
                            if !present(mondes, w, inner2) {
                                ajoute(mondes, w, (**inner2).clone())?;
                                change = true;
                            }
                        } else if let Formule::Imp(x, y) = &**inner {
                            for g in [(**x).clone(), non((**y).clone())] {
                                if !present(mondes, w, &g) {
                                    ajoute(mondes, w, g)?;
                                    change = true;
                                }
                            }
                        } else if let Formule::Diamant(g) = &**inner {
                            let besoin = non((**g).clone());
                            let cibles: Vec<usize> = mondes.keys().cloned().filter(|x| voit(w, *x)).collect();
                            for v in cibles {
                                if !present(mondes, v, &besoin) {
                                    ajoute(mondes, v, besoin.clone())?;
                                    change = true;
                                }
                            }
                        } else if let Formule::Boite(g) = &**inner {
                            let besoin = non((**g).clone());
                            let vu = mondes.keys().cloned().filter(|x| voit(w, *x))
                                .any(|v| present(mondes, v, &besoin));
                            if !vu {
                                if *mondes.keys().max().expect("mondes non vide") >= max_profondeur {
                                    return Ok(Fin::Incomplet);
                                }
                                let nouveau = *mondes.keys().max().expect("mondes non vide") + 1;
                                let mut ensemble: HashSet<Formule> = HashSet::new();
                                ensemble.insert(besoin.clone());
                                mondes.insert(nouveau, ensemble);
                                change = true;
                            }
                        }
                    }
                    Formule::Et(x, y) => {
                        for g in [(**x).clone(), (**y).clone()] {
                            if !present(mondes, w, &g) {
                                ajoute(mondes, w, g)?;
                                change = true;
                            }
                        }
                    }
                    Formule::Imp(x, y) => {
                        // Deux ajouts dans la meme visite (modus ponens puis
                        // contraposition), chacun avec son propre test.
                        if present(mondes, w, x) && !present(mondes, w, y) {
                            ajoute(mondes, w, (**y).clone())?;
                            change = true;
                        }
                        let nx = non((**x).clone());
                        let ny = non((**y).clone());
                        if present(mondes, w, &ny) && !present(mondes, w, &nx) {
                            ajoute(mondes, w, nx)?;
                            change = true;
                        }
                    }
                    Formule::Boite(g) => {
                        let cibles: Vec<usize> = mondes.keys().cloned().filter(|x| voit(w, *x)).collect();
                        for v in cibles {
                            if !present(mondes, v, g) {
                                ajoute(mondes, v, (**g).clone())?;
                                change = true;
                            }
                        }
                    }
                    Formule::Diamant(g) => {
                        let besoin = (**g).clone();
                        let vu = mondes.keys().cloned().filter(|x| voit(w, *x))
                            .any(|v| present(mondes, v, &besoin));
                        if !vu {
                            if *mondes.keys().max().expect("mondes non vide") >= max_profondeur {
                                return Ok(Fin::Incomplet);
                            }
                            let nouveau = *mondes.keys().max().expect("mondes non vide") + 1;
                            let mut ensemble: HashSet<Formule> = HashSet::new();
                            ensemble.insert(besoin.clone());
                            mondes.insert(nouveau, ensemble);
                            change = true;
                        }
                    }
                    Formule::Atome(_) => {}
                }
            }
        }
        if !change {
            return Ok(Fin::Sat);
        }
    }
}

/// Saturation complete : `Unsat` porte la contradiction, `Sat` le modele fini.
#[allow(dead_code)]
fn refute(premisses: &[Formule], max_profondeur: usize) -> Resultat {
    let mut ensemble: HashSet<Formule> = HashSet::new();
    for p in premisses { ensemble.insert(p.clone()); }
    let mut mondes: BTreeMap<usize, HashSet<Formule>> = BTreeMap::new();
    mondes.insert(0, ensemble);
    match saturer(&mut mondes, max_profondeur) {
        Ok(Fin::Sat) => Resultat::Sat(mondes),
        Ok(Fin::Incomplet) => Resultat::Incomplet,
        Err(c) => Resultat::Unsat(c),
    }
}

println!("Derouleur defini : refute(premisses) -> (statut, info).");
println!("Statuts : unsat (contradiction, info = monde), sat (modele fini), incomplet (borne).");

Derouleur defini : refute(premisses) -> (statut, info).


Statuts : unsat (contradiction, info = monde), sat (modele fini), incomplet (borne).


### Lecture du dérouleur

Trois statuts, et aucun des trois n'est un « vrai » :

| statut | sens | ce qu'on peut en dire |
|---|---|---|
| `unsat` | l'atome et sa négation se sont rencontrés au monde `monde` de la contradiction | **réfutation** : l'hypothèse de départ est incompatible avec les prémisses |
| `sat` | saturation sans contradiction | un **modèle fini** des prémisses : le comportement observé est *possible*, pas *forcé* |
| `incomplet` | borne atteinte, obligation encore ouverte | ni cohérent, ni contradictoire : **on ne sait pas** |

Deux limites à garder en tête avant de lire le tableau suivant.

1. **`sat` n'est pas une preuve de cohérence.** Le modèle trouvé est un modèle du *cadre employé* — une chaîne transitive finie de mondes. Un autre cadre pourrait fermer la réfutation ; un modèle, c'est une possibilité, pas un théorème.
2. **Le dérouleur n'encode pas la bonne-fondation.** Les cadres de GL sont transitifs **et** bien fondés : aucune chaîne infinie de mondes accessibles. Ici la visibilité est la simple chaîne `i < j` sur les mondes déjà créés, sans aucun axiome de bonne-fondation. Quand une obligation ouvre un monde de plus à chaque passe, le dérouleur en ouvre indéfiniment et bute sur sa borne. C'est une propriété du **dérouleur**, pas une découverte sur Löb — et c'est précisément ce qui rend le cas löbien `INCOMPLET` à la section suivante.

In [4]:
// --- Les quatre jeux de premisses du post ---
use std::collections::{BTreeMap, HashSet};

/// AST des formules modales : la recursion est portee par `Box`.
/// `Clone + PartialEq + Eq + Hash` reproduit la dataclass `frozen` Python :
/// l'egalite et le hachage sont structurels.
#[derive(Clone, PartialEq, Eq, Hash)]
#[allow(dead_code)]
enum Formule {
    Atome(String),
    Non(Box<Formule>),
    Imp(Box<Formule>, Box<Formule>),
    Et(Box<Formule>, Box<Formule>),
    Boite(Box<Formule>),
    Diamant(Box<Formule>),
}

/// Un atome du langage modal.
#[allow(dead_code)]
fn a(nom: &str) -> Formule { Formule::Atome(nom.to_string()) }
/// La negation d'une formule.
#[allow(dead_code)]
fn non(f: Formule) -> Formule { Formule::Non(Box::new(f)) }
/// L'implication.
#[allow(dead_code)]
fn imp(a: Formule, b: Formule) -> Formule { Formule::Imp(Box::new(a), Box::new(b)) }
/// La boite (necessite) : `Box` est un mot-cle, d'ou le nom `box_`.
#[allow(dead_code)]
fn box_(f: Formule) -> Formule { Formule::Boite(Box::new(f)) }
/// Le diamant (possibilite).
#[allow(dead_code)]
fn dia(f: Formule) -> Formule { Formule::Diamant(Box::new(f)) }
/// L'equivalence : `f <-> g`.
#[allow(dead_code)]
fn equiv(f: Formule, g: Formule) -> Formule {
    Formule::Et(Box::new(imp(f.clone(), g.clone())), Box::new(imp(g, f)))
}

/// Impresseur unique des formules : jamais `{:?}` (le `repr` Python n'a pas
/// d'equivalent lisible, et `show` est le seul imprimeur du carnet).
#[allow(dead_code)]
fn show(f: &Formule) -> String {
    match f {
        Formule::Atome(nom) => nom.clone(),
        Formule::Non(g) => format!("non {}", show(g)),
        Formule::Imp(a, b) => format!("({} -> {})", show(a), show(b)),
        Formule::Et(a, b) => format!("({} et {})", show(a), show(b)),
        Formule::Boite(g) => format!("[]{}", show(g)),
        Formule::Diamant(g) => format!("<>{}", show(g)),
    }
}

/// L'exception Python devient une valeur : le rolleur la propage par `Result`.
#[allow(dead_code)]
struct Contradiction { monde: usize, f: Formule }

/// Chaine transitive : le monde i voit tout monde cree apres lui.
#[allow(dead_code)]
fn voit(i: usize, j: usize) -> bool { i < j }

/// La formule est-elle deja presente au monde `w` ?
#[allow(dead_code)]
fn present(mondes: &BTreeMap<usize, HashSet<Formule>>, w: usize, f: &Formule) -> bool {
    mondes.get(&w).expect("monde present").contains(f)
}

/// Ajout d'une formule au monde `w`, avec detection de contradiction.
///
/// Asymetrie mesuree : si `non(f)` est deja present, l'erreur porte `f` ;
/// si `f` est `non(g)` et `g` est present, l'erreur porte `g` (la formule
/// interne, positive). Inverser les deux branches changerait les deux
/// messages affiches par la table.
#[allow(dead_code)]
fn ajoute(mondes: &mut BTreeMap<usize, HashSet<Formule>>, w: usize, f: Formule) -> Result<(), Contradiction> {
    if present(mondes, w, &Formule::Non(Box::new(f.clone()))) {
        return Err(Contradiction { monde: w, f: f.clone() });
    }
    if let Formule::Non(g) = &f {
        if present(mondes, w, g) {
            return Err(Contradiction { monde: w, f: (**g).clone() });
        }
    }
    mondes.get_mut(&w).expect("monde present").insert(f);
    Ok(())
}

/// Fin de la saturation : modele fini, ou borne atteinte.
#[allow(dead_code)]
enum Fin { Sat, Incomplet }

/// Verdict du rouleau : contradiction, modele fini, ou borne atteinte.
#[allow(dead_code)]
enum Resultat {
    Unsat(Contradiction),
    Sat(BTreeMap<usize, HashSet<Formule>>),
    Incomplet,
}

/// Saturation : on decompose les formules jusqu'a ce qu'aucune passe ne change rien.
///
/// L'ordre des bras suit celui des `if/elif` de la source : `non (non f)`,
/// `et`, `imp`, `non (a -> b)`, `box`, `non (diamant)`, `non (box)`,
/// `diamant`. `non (a et b)` ne tombe dans aucun bras : c'est la limite
/// assumee du carnet.
#[allow(dead_code)]
fn saturer(mondes: &mut BTreeMap<usize, HashSet<Formule>>, max_profondeur: usize) -> Result<Fin, Contradiction> {
    loop {
        let mut change = false;
        // Instantanés : Python itere sur des copies (`sorted(list(mondes))`),
        // et mute le dict pendant la passe.
        let cles: Vec<usize> = mondes.keys().cloned().collect();
        for w in cles {
            let mut fs: Vec<Formule> = mondes.get(&w).expect("monde present").iter().cloned().collect();
            fs.sort_by_key(|f| show(f));
            for f in fs {
                match &f {
                    Formule::Non(inner) => {
                        if let Formule::Non(inner2) = &**inner {
                            if !present(mondes, w, inner2) {
                                ajoute(mondes, w, (**inner2).clone())?;
                                change = true;
                            }
                        } else if let Formule::Imp(x, y) = &**inner {
                            for g in [(**x).clone(), non((**y).clone())] {
                                if !present(mondes, w, &g) {
                                    ajoute(mondes, w, g)?;
                                    change = true;
                                }
                            }
                        } else if let Formule::Diamant(g) = &**inner {
                            let besoin = non((**g).clone());
                            let cibles: Vec<usize> = mondes.keys().cloned().filter(|x| voit(w, *x)).collect();
                            for v in cibles {
                                if !present(mondes, v, &besoin) {
                                    ajoute(mondes, v, besoin.clone())?;
                                    change = true;
                                }
                            }
                        } else if let Formule::Boite(g) = &**inner {
                            let besoin = non((**g).clone());
                            let vu = mondes.keys().cloned().filter(|x| voit(w, *x))
                                .any(|v| present(mondes, v, &besoin));
                            if !vu {
                                if *mondes.keys().max().expect("mondes non vide") >= max_profondeur {
                                    return Ok(Fin::Incomplet);
                                }
                                let nouveau = *mondes.keys().max().expect("mondes non vide") + 1;
                                let mut ensemble: HashSet<Formule> = HashSet::new();
                                ensemble.insert(besoin.clone());
                                mondes.insert(nouveau, ensemble);
                                change = true;
                            }
                        }
                    }
                    Formule::Et(x, y) => {
                        for g in [(**x).clone(), (**y).clone()] {
                            if !present(mondes, w, &g) {
                                ajoute(mondes, w, g)?;
                                change = true;
                            }
                        }
                    }
                    Formule::Imp(x, y) => {
                        // Deux ajouts dans la meme visite (modus ponens puis
                        // contraposition), chacun avec son propre test.
                        if present(mondes, w, x) && !present(mondes, w, y) {
                            ajoute(mondes, w, (**y).clone())?;
                            change = true;
                        }
                        let nx = non((**x).clone());
                        let ny = non((**y).clone());
                        if present(mondes, w, &ny) && !present(mondes, w, &nx) {
                            ajoute(mondes, w, nx)?;
                            change = true;
                        }
                    }
                    Formule::Boite(g) => {
                        let cibles: Vec<usize> = mondes.keys().cloned().filter(|x| voit(w, *x)).collect();
                        for v in cibles {
                            if !present(mondes, v, g) {
                                ajoute(mondes, v, (**g).clone())?;
                                change = true;
                            }
                        }
                    }
                    Formule::Diamant(g) => {
                        let besoin = (**g).clone();
                        let vu = mondes.keys().cloned().filter(|x| voit(w, *x))
                            .any(|v| present(mondes, v, &besoin));
                        if !vu {
                            if *mondes.keys().max().expect("mondes non vide") >= max_profondeur {
                                return Ok(Fin::Incomplet);
                            }
                            let nouveau = *mondes.keys().max().expect("mondes non vide") + 1;
                            let mut ensemble: HashSet<Formule> = HashSet::new();
                            ensemble.insert(besoin.clone());
                            mondes.insert(nouveau, ensemble);
                            change = true;
                        }
                    }
                    Formule::Atome(_) => {}
                }
            }
        }
        if !change {
            return Ok(Fin::Sat);
        }
    }
}

/// Saturation complete : `Unsat` porte la contradiction, `Sat` le modele fini.
#[allow(dead_code)]
fn refute(premisses: &[Formule], max_profondeur: usize) -> Resultat {
    let mut ensemble: HashSet<Formule> = HashSet::new();
    for p in premisses { ensemble.insert(p.clone()); }
    let mut mondes: BTreeMap<usize, HashSet<Formule>> = BTreeMap::new();
    mondes.insert(0, ensemble);
    match saturer(&mut mondes, max_profondeur) {
        Ok(Fin::Sat) => Resultat::Sat(mondes),
        Ok(Fin::Incomplet) => Resultat::Incomplet,
        Err(c) => Resultat::Unsat(c),
    }
}

/// Plus petite borne qui ferme ; `(None, Incomplet)` si aucune ne ferme.
///
/// Rust n'a pas d'argument par defaut : la borne est passee explicitement.
#[allow(dead_code)]
fn profondeur_minimale(premisses: &[Formule], borne: usize) -> (Option<usize>, Resultat) {
    for k in 1..=borne {
        let resultat = refute(premisses, k);
        if !matches!(resultat, Resultat::Incomplet) {
            return (Some(k), resultat);
        }
    }
    (None, Resultat::Incomplet)
}

/// Le statut, sous sa forme textuelle.
#[allow(dead_code)]
fn nom_statut(r: &Resultat) -> &'static str {
    match r {
        Resultat::Unsat(_) => "unsat",
        Resultat::Sat(_) => "sat",
        Resultat::Incomplet => "incomplet",
    }
}

/// Repr Python d'une liste de chaines : ['a', 'b'].
/// Jamais `{:?}` sur un `Vec<String>` : Rust ecrirait ["a", "b"].
#[allow(dead_code)]
fn py_list_repr(items: &[String]) -> String {
    let elements: Vec<String> = items.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", elements.join(", "))
}

let p_at = a("P");
let q_at = a("Q");
let h_at = a("H");

let cond_payor = equiv(p_at.clone(), box_(imp(box_(p_at.clone()), q_at.clone())));   // P coopere ssi P prouve []([]P -> Q)
let cond_payor_q = equiv(q_at.clone(), box_(imp(box_(q_at.clone()), p_at.clone()))); // la meme definition, vue de Q
let cond_coop = box_(q_at.clone());                                                // CooperateBot : coopere toujours
let cond_defect = box_(non(q_at.clone()));                                         // DefectBot : trahit toujours
let cond_loeb = equiv(h_at.clone(), box_(h_at.clone()));                           // phrase de Henkin : H <-> []H

// Pourquoi les definitions sont AUSSI posees sous une boite (box(COND_...)) :
// la refutation de ~P se joue dans le monde w1, ou l'on simule l'adversaire.
// Pour que cette simulation raisonne avec SA definition, celle-ci doit etre
// accessible dans les successeurs -- c'est la premisse supplementaire que le
// fil reclame explicitement ("repeating the description of myself but wrapped
// in a provability modality so it's accessible in the simulations", 8mo).

// L'ordre des lignes est celui d'insertion du dict Python : un `BTreeMap`
// trierait les lignes (`Lob` en tete) et l'ordre est porteur.
let cas: Vec<(&str, Vec<Formule>)> = vec![
    ("P vs CooperateBot", vec![cond_payor.clone(), box_(cond_payor.clone()), cond_coop.clone(), non(p_at.clone())]),
    ("P vs Payorian Q", vec![cond_payor.clone(), box_(cond_payor.clone()), cond_payor_q.clone(), box_(cond_payor_q.clone()), non(p_at.clone())]),
    ("P vs DefectBot", vec![cond_payor.clone(), box_(cond_payor.clone()), cond_defect.clone(), non(p_at.clone())]),
    ("Lob : H <-> []H", vec![cond_loeb.clone(), box_(cond_loeb.clone()), non(h_at.clone())]),
];

println!("{:<20} {:<10} {:<7} detail", "jeu de premisses", "statut", "borne");
println!("{}", "-".repeat(78));
for (nom, premisses) in &cas {
    let (borne, resultat) = profondeur_minimale(premisses, 6);
    let statut = nom_statut(&resultat);
    let detail = match &resultat {
        Resultat::Unsat(c) => format!("contradiction en {} : {}", c.monde, show(&c.f)),
        Resultat::Sat(m) => format!("modele a {} mondes", m.len()),
        Resultat::Incomplet => "borne atteinte, obligation ouverte".to_string(),
    };
    // `Option<usize>` n'implemente pas `Display` : Python imprimait `None`.
    let borne_texte = match borne { Some(k) => k.to_string(), None => "None".to_string() };
    println!("{:<20} {:<10} {:<7} {}", nom, statut, borne_texte, detail);
}

// --- Controles : les trois profondeurs annoncees par le post, plus la frontiere ---
let (b1, r1) = profondeur_minimale(&cas[0].1, 6);
assert_eq!((nom_statut(&r1), b1), ("unsat", Some(1)));   // un seul monde ajoute
let (b2, r2) = profondeur_minimale(&cas[1].1, 6);
assert_eq!((nom_statut(&r2), b2), ("unsat", Some(2)));   // deux mondes : c'est le "2 au lieu de 4"
let (_b3, r3) = profondeur_minimale(&cas[2].1, 6);
assert_eq!(nom_statut(&r3), "sat");
let m3 = match &r3 {
    Resultat::Sat(m) => m.clone(),
    _ => BTreeMap::new(),
};
assert_eq!(m3.len(), 2);
let (_b4, r4) = profondeur_minimale(&cas[3].1, 6);
assert_eq!(nom_statut(&r4), "incomplet");

// Lecture du modele de P vs DefectBot : le monde 1 porte []P et ~Q.
// L'atome P n'y figure pas -- c'est la biconditionnelle, verifiee dans la
// cellule suivante, qui l'y rend vrai.
assert!(present(&m3, 1, &box_(p_at.clone())));
assert!(present(&m3, 1, &non(q_at.clone())));
println!();
let mut montre: Vec<String> = m3.get(&1).expect("monde 1").iter().map(|f| show(f)).collect();
montre.sort();
println!("P vs DefectBot : monde 1 = {}", py_list_repr(&montre));
println!("Controles passes : 1 monde, 2 mondes, sat a 2 mondes, incomplet.");

jeu de premisses     statut     borne   detail


------------------------------------------------------------------------------


P vs CooperateBot    unsat      1       contradiction en 1 : Q


P vs Payorian Q      unsat      2       contradiction en 2 : P


P vs DefectBot       sat        1       modele a 2 mondes


Lob : H <-> []H      incomplet  None    borne atteinte, obligation ouverte


P vs DefectBot : monde 1 = ['((P -> []([]P -> Q)) et ([]([]P -> Q) -> P))', '(P -> []([]P -> Q))', '([]([]P -> Q) -> P)', '[]P', 'non ([]P -> Q)', 'non Q']


Controles passes : 1 monde, 2 mondes, sat a 2 mondes, incomplet.


### Vérifier un `sat` : les deux moteurs doivent s'accorder

Un `sat` n'est pas une ligne de console, c'est l'affirmation qu'un modèle existe. On la vérifie en reconstruisant un `Modele` à partir des mondes saturés, puis en repassant **chaque prémisse** au moteur de forcing de la section 2 — un moteur écrit indépendamment, pour un autre carnet.

La reconstruction prend une convention, et il faut la nommer : un atome est vrai dans un monde **sauf si** sa négation y figure. C'est une fermeture par défaut, pas une conséquence du dérouleur. Si les prémisses ne tiennent pas sous cette convention, le `sat` est faux et l'`assert` le signale ; si elles tiennent, on a bien un modèle, et les deux moteurs s'accordent.

In [5]:
// --- Un "sat" se verifie : on repasse le modele au moteur de la section 2 ---
use std::collections::{BTreeMap, HashMap, HashSet};

/// AST des formules modales : la recursion est portee par `Box`.
/// `Clone + PartialEq + Eq + Hash` reproduit la dataclass `frozen` Python :
/// l'egalite et le hachage sont structurels.
#[derive(Clone, PartialEq, Eq, Hash)]
#[allow(dead_code)]
enum Formule {
    Atome(String),
    Non(Box<Formule>),
    Imp(Box<Formule>, Box<Formule>),
    Et(Box<Formule>, Box<Formule>),
    Boite(Box<Formule>),
    Diamant(Box<Formule>),
}

/// Un atome du langage modal.
#[allow(dead_code)]
fn a(nom: &str) -> Formule { Formule::Atome(nom.to_string()) }
/// La negation d'une formule.
#[allow(dead_code)]
fn non(f: Formule) -> Formule { Formule::Non(Box::new(f)) }
/// L'implication.
#[allow(dead_code)]
fn imp(a: Formule, b: Formule) -> Formule { Formule::Imp(Box::new(a), Box::new(b)) }
/// La boite (necessite) : `Box` est un mot-cle, d'ou le nom `box_`.
#[allow(dead_code)]
fn box_(f: Formule) -> Formule { Formule::Boite(Box::new(f)) }
/// Le diamant (possibilite).
#[allow(dead_code)]
fn dia(f: Formule) -> Formule { Formule::Diamant(Box::new(f)) }
/// L'equivalence : `f <-> g`.
#[allow(dead_code)]
fn equiv(f: Formule, g: Formule) -> Formule {
    Formule::Et(Box::new(imp(f.clone(), g.clone())), Box::new(imp(g, f)))
}

/// Impresseur unique des formules : jamais `{:?}` (le `repr` Python n'a pas
/// d'equivalent lisible, et `show` est le seul imprimeur du carnet).
#[allow(dead_code)]
fn show(f: &Formule) -> String {
    match f {
        Formule::Atome(nom) => nom.clone(),
        Formule::Non(g) => format!("non {}", show(g)),
        Formule::Imp(a, b) => format!("({} -> {})", show(a), show(b)),
        Formule::Et(a, b) => format!("({} et {})", show(a), show(b)),
        Formule::Boite(g) => format!("[]{}", show(g)),
        Formule::Diamant(g) => format!("<>{}", show(g)),
    }
}

/// L'exception Python devient une valeur : le rolleur la propage par `Result`.
#[allow(dead_code)]
struct Contradiction { monde: usize, f: Formule }

/// Chaine transitive : le monde i voit tout monde cree apres lui.
#[allow(dead_code)]
fn voit(i: usize, j: usize) -> bool { i < j }

/// La formule est-elle deja presente au monde `w` ?
#[allow(dead_code)]
fn present(mondes: &BTreeMap<usize, HashSet<Formule>>, w: usize, f: &Formule) -> bool {
    mondes.get(&w).expect("monde present").contains(f)
}

/// Ajout d'une formule au monde `w`, avec detection de contradiction.
///
/// Asymetrie mesuree : si `non(f)` est deja present, l'erreur porte `f` ;
/// si `f` est `non(g)` et `g` est present, l'erreur porte `g` (la formule
/// interne, positive). Inverser les deux branches changerait les deux
/// messages affiches par la table.
#[allow(dead_code)]
fn ajoute(mondes: &mut BTreeMap<usize, HashSet<Formule>>, w: usize, f: Formule) -> Result<(), Contradiction> {
    if present(mondes, w, &Formule::Non(Box::new(f.clone()))) {
        return Err(Contradiction { monde: w, f: f.clone() });
    }
    if let Formule::Non(g) = &f {
        if present(mondes, w, g) {
            return Err(Contradiction { monde: w, f: (**g).clone() });
        }
    }
    mondes.get_mut(&w).expect("monde present").insert(f);
    Ok(())
}

/// Fin de la saturation : modele fini, ou borne atteinte.
#[allow(dead_code)]
enum Fin { Sat, Incomplet }

/// Verdict du rouleau : contradiction, modele fini, ou borne atteinte.
#[allow(dead_code)]
enum Resultat {
    Unsat(Contradiction),
    Sat(BTreeMap<usize, HashSet<Formule>>),
    Incomplet,
}

/// Saturation : on decompose les formules jusqu'a ce qu'aucune passe ne change rien.
///
/// L'ordre des bras suit celui des `if/elif` de la source : `non (non f)`,
/// `et`, `imp`, `non (a -> b)`, `box`, `non (diamant)`, `non (box)`,
/// `diamant`. `non (a et b)` ne tombe dans aucun bras : c'est la limite
/// assumee du carnet.
#[allow(dead_code)]
fn saturer(mondes: &mut BTreeMap<usize, HashSet<Formule>>, max_profondeur: usize) -> Result<Fin, Contradiction> {
    loop {
        let mut change = false;
        // Instantanés : Python itere sur des copies (`sorted(list(mondes))`),
        // et mute le dict pendant la passe.
        let cles: Vec<usize> = mondes.keys().cloned().collect();
        for w in cles {
            let mut fs: Vec<Formule> = mondes.get(&w).expect("monde present").iter().cloned().collect();
            fs.sort_by_key(|f| show(f));
            for f in fs {
                match &f {
                    Formule::Non(inner) => {
                        if let Formule::Non(inner2) = &**inner {
                            if !present(mondes, w, inner2) {
                                ajoute(mondes, w, (**inner2).clone())?;
                                change = true;
                            }
                        } else if let Formule::Imp(x, y) = &**inner {
                            for g in [(**x).clone(), non((**y).clone())] {
                                if !present(mondes, w, &g) {
                                    ajoute(mondes, w, g)?;
                                    change = true;
                                }
                            }
                        } else if let Formule::Diamant(g) = &**inner {
                            let besoin = non((**g).clone());
                            let cibles: Vec<usize> = mondes.keys().cloned().filter(|x| voit(w, *x)).collect();
                            for v in cibles {
                                if !present(mondes, v, &besoin) {
                                    ajoute(mondes, v, besoin.clone())?;
                                    change = true;
                                }
                            }
                        } else if let Formule::Boite(g) = &**inner {
                            let besoin = non((**g).clone());
                            let vu = mondes.keys().cloned().filter(|x| voit(w, *x))
                                .any(|v| present(mondes, v, &besoin));
                            if !vu {
                                if *mondes.keys().max().expect("mondes non vide") >= max_profondeur {
                                    return Ok(Fin::Incomplet);
                                }
                                let nouveau = *mondes.keys().max().expect("mondes non vide") + 1;
                                let mut ensemble: HashSet<Formule> = HashSet::new();
                                ensemble.insert(besoin.clone());
                                mondes.insert(nouveau, ensemble);
                                change = true;
                            }
                        }
                    }
                    Formule::Et(x, y) => {
                        for g in [(**x).clone(), (**y).clone()] {
                            if !present(mondes, w, &g) {
                                ajoute(mondes, w, g)?;
                                change = true;
                            }
                        }
                    }
                    Formule::Imp(x, y) => {
                        // Deux ajouts dans la meme visite (modus ponens puis
                        // contraposition), chacun avec son propre test.
                        if present(mondes, w, x) && !present(mondes, w, y) {
                            ajoute(mondes, w, (**y).clone())?;
                            change = true;
                        }
                        let nx = non((**x).clone());
                        let ny = non((**y).clone());
                        if present(mondes, w, &ny) && !present(mondes, w, &nx) {
                            ajoute(mondes, w, nx)?;
                            change = true;
                        }
                    }
                    Formule::Boite(g) => {
                        let cibles: Vec<usize> = mondes.keys().cloned().filter(|x| voit(w, *x)).collect();
                        for v in cibles {
                            if !present(mondes, v, g) {
                                ajoute(mondes, v, (**g).clone())?;
                                change = true;
                            }
                        }
                    }
                    Formule::Diamant(g) => {
                        let besoin = (**g).clone();
                        let vu = mondes.keys().cloned().filter(|x| voit(w, *x))
                            .any(|v| present(mondes, v, &besoin));
                        if !vu {
                            if *mondes.keys().max().expect("mondes non vide") >= max_profondeur {
                                return Ok(Fin::Incomplet);
                            }
                            let nouveau = *mondes.keys().max().expect("mondes non vide") + 1;
                            let mut ensemble: HashSet<Formule> = HashSet::new();
                            ensemble.insert(besoin.clone());
                            mondes.insert(nouveau, ensemble);
                            change = true;
                        }
                    }
                    Formule::Atome(_) => {}
                }
            }
        }
        if !change {
            return Ok(Fin::Sat);
        }
    }
}

/// Saturation complete : `Unsat` porte la contradiction, `Sat` le modele fini.
#[allow(dead_code)]
fn refute(premisses: &[Formule], max_profondeur: usize) -> Resultat {
    let mut ensemble: HashSet<Formule> = HashSet::new();
    for p in premisses { ensemble.insert(p.clone()); }
    let mut mondes: BTreeMap<usize, HashSet<Formule>> = BTreeMap::new();
    mondes.insert(0, ensemble);
    match saturer(&mut mondes, max_profondeur) {
        Ok(Fin::Sat) => Resultat::Sat(mondes),
        Ok(Fin::Incomplet) => Resultat::Incomplet,
        Err(c) => Resultat::Unsat(c),
    }
}

/// Plus petite borne qui ferme ; `(None, Incomplet)` si aucune ne ferme.
///
/// Rust n'a pas d'argument par defaut : la borne est passee explicitement.
#[allow(dead_code)]
fn profondeur_minimale(premisses: &[Formule], borne: usize) -> (Option<usize>, Resultat) {
    for k in 1..=borne {
        let resultat = refute(premisses, k);
        if !matches!(resultat, Resultat::Incomplet) {
            return (Some(k), resultat);
        }
    }
    (None, Resultat::Incomplet)
}

/// Les atomes qui apparaissent dans une formule.
#[allow(dead_code)]
fn atomes(f: &Formule) -> HashSet<String> {
    match f {
        Formule::Atome(nom) => { let mut s = HashSet::new(); s.insert(nom.clone()); s }
        Formule::Non(g) => atomes(g),
        Formule::Imp(x, y) => { let mut s = atomes(x); s.extend(atomes(y)); s }
        Formule::Et(x, y) => { let mut s = atomes(x); s.extend(atomes(y)); s }
        Formule::Boite(g) => atomes(g),
        Formule::Diamant(g) => atomes(g),
    }
}

/// Cadre de Kripke reconstruit des mondes satures.
#[allow(dead_code)]
struct Modele {
    mondes: Vec<usize>,
    relation: HashSet<(usize, usize)>,
    valuation: HashMap<String, HashSet<usize>>,
}

/// Les mondes visibles depuis `x`.
#[allow(dead_code)]
fn successeurs(m: &Modele, x: usize) -> Vec<usize> {
    m.relation.iter().filter(|(a, _)| *a == x).map(|(_, y)| *y).collect()
}

/// Forcing : `Boite` = tous les successeurs, `Diamant` = au moins un.
#[allow(dead_code)]
fn satisfait(m: &Modele, x: usize, f: &Formule) -> bool {
    match f {
        Formule::Atome(nom) => m.valuation.get(nom).expect("atome du modele").contains(&x),
        Formule::Non(g) => !satisfait(m, x, g),
        Formule::Imp(a, b) => !satisfait(m, x, a) || satisfait(m, x, b),
        Formule::Et(a, b) => satisfait(m, x, a) && satisfait(m, x, b),
        Formule::Boite(g) => successeurs(m, x).iter().all(|y| satisfait(m, *y, g)),
        Formule::Diamant(g) => successeurs(m, x).iter().any(|y| satisfait(m, *y, g)),
    }
}

/// Modele reconstruit des mondes satures.
///
/// Fermeture par defaut : un atome est vrai en w sauf si non(Atome) y figure.
/// La relation est la chaine transitive i < j, celle du derouleur.
#[allow(dead_code)]
fn vers_modele(mondes: &BTreeMap<usize, HashSet<Formule>>) -> Modele {
    let mut noms: HashSet<String> = HashSet::new();
    for fs in mondes.values() {
        for f in fs { noms.extend(atomes(f)); }
    }
    let mut valuation: HashMap<String, HashSet<usize>> = HashMap::new();
    for n in &noms {
        let faux = non(Formule::Atome(n.clone()));
        let ensemble: HashSet<usize> = mondes.iter()
            .filter(|(_, fs)| !fs.contains(&faux))
            .map(|(w, _)| *w)
            .collect();
        valuation.insert(n.clone(), ensemble);
    }
    let mut relation: HashSet<(usize, usize)> = HashSet::new();
    for i in mondes.keys() {
        for j in mondes.keys() {
            if i < j { relation.insert((*i, *j)); }
        }
    }
    let mut liste: Vec<usize> = mondes.keys().cloned().collect();
    liste.sort();
    Modele { mondes: liste, relation, valuation }
}

let p_at = a("P");
let q_at = a("Q");
let cond_payor = equiv(p_at.clone(), box_(imp(box_(p_at.clone()), q_at.clone())));
let cond_defect = box_(non(q_at.clone()));
// Le meme jeu de premisses que `CAS["P vs DefectBot"]` de la cellule 10.
let cas_defect: Vec<Formule> = vec![
    cond_payor.clone(),
    box_(cond_payor.clone()),
    cond_defect.clone(),
    non(p_at.clone()),
];

let (_b3, r3) = profondeur_minimale(&cas_defect, 6);
let m3 = match &r3 {
    Resultat::Sat(m) => m.clone(),
    _ => BTreeMap::new(),
};
let m3_modele = vers_modele(&m3);
let verdicts: Vec<bool> = cas_defect.iter().map(|p| satisfait(&m3_modele, 0, p)).collect();
for (p, v) in cas_defect.iter().zip(verdicts.iter()) {
    println!("  {:4}  {}", if *v { "vrai" } else { "FAUX" }, show(p));
}
assert!(verdicts.iter().all(|v| *v), "le modele ne satisfait pas les premisses : le 'sat' serait faux");

println!();
println!("Modele valide : les quatre premisses tiennent en w0.");
println!("P est vrai en w1 par la biconditionnelle, pas par propagation de []P :");
println!("le monde 1 est terminal, donc []P n'y contraint aucun successeur.");

  vrai  ((P -> []([]P -> Q)) et ([]([]P -> Q) -> P))


  vrai  []((P -> []([]P -> Q)) et ([]([]P -> Q) -> P))


  vrai  []non Q


  vrai  non P


Modele valide : les quatre premisses tiennent en w0.


P est vrai en w1 par la biconditionnelle, pas par propagation de []P :


le monde 1 est terminal, donc []P n'y contraint aucun successeur.


### Lecture du tableau

**Une prémisse qui n'est pas un détail.** Les deux définitions sont posées à la racine **et** sous une boîte. Sans la boîte, la définition de l'adversaire n'atteint pas `w1`, la simulation ne sait pas quel bot elle simule, et la réfutation rend un `sat` trompeur. Le fil le formule lui-même : *repeating the description of myself but wrapped in a provability modality so it's accessible in the simulations*. C'est une prémisse du modèle, pas un ajustement du code — et c'est le genre d'oubli qui fait passer un carnet pour une preuve.

**P vs CooperateBot — un monde.** `□Q` à la racine place `Q` dans le seul monde créé, où `¬Q` était déjà posé par la négation de la clause de Payor. La contradiction est **interne** au monde 1 : aucun enchaînement n'est nécessaire.

**P vs Payorian Q — deux mondes.** C'est le résultat central. En `w1` on a `□P` et `¬Q`. Le `¬Q` attaque la définition de Q, qui ouvre `w2` avec `¬P` ; et le `□P` de `w1` place `P` dans `w2`. La contradiction naît **entre les deux définitions**, dans un monde que chacune atteint par un chemin différent. Deux mondes ajoutés, pas quatre.

**P vs DefectBot — saturé, deux mondes.** Le modèle est petit : `w0` porte `¬P`, `w1` porte `□P` et `¬Q`. Tout tient, et le point qui fait tenir est que **`w1` ne voit rien** : `□(□P→Q)` y est vacuement vrai, comme la boîte sur un monde terminal de la section 2. Ce `sat` n'est pas un accident de programmation : la cellule suivante le **repasse au moteur de la section 2** et vérifie que les quatre prémisses tiennent. C'est la **frontière de l'hypothèse de cohérence** — la réfutation seule ne conclut pas que le FairBot payorien trahit contre DefectBot. Il faut lui ajouter une hypothèse de cohérence sur son propre système de preuve, et c'est écrit dans le post comme une réserve, pas comme un théorème.

**Löb — incomplet.** La phrase de Henkin `H ↔ □H` n'ouvre pas une contradiction : elle ouvre un monde de plus à chaque passe, indéfiniment. Le dérouleur rend `incomplet`, ce qui est la réponse **honnête** de cet instrument-là. Un prouveur de GL fermerait, parce qu'il dispose de l'axiome de Löb et de la bonne-fondation ; notre dérouleur ne les a pas.

## 4. Le tournoi, en jouets

La section 3 a réfuté des formules. Il reste à voir ce que ces réfutations **disent du jeu**. On reprend la convention du carnet `Dynamics-04-Open-Source-Game-Theory-Python` — un bot rend `"C"` ou `"D"` après avoir lu quelque chose de son adversaire — en remplaçant la lecture du code source par une **déclaration** en clair : chaque bot publie sa règle, et c'est ce texte que les autres lisent. Une caricature assumée, et il faut dire exactement ce qu'elle vaut :

- le tournoi **illustre** les issues que l'analyse de Kripke contraint ;
- il ne les **démontre pas**. Sur le cas `P vs DefectBot`, l'analyse rend `sat` : deux issues sont compatibles avec les prémisses, et le jouet en choisit une par construction. Ce choix n'est pas une dérivation, c'est une règle d'écriture.

La déclaration joue le rôle de la définition de l'adversaire dans une simulation modale : le bot ne voit pas la fonction, il voit ce que l'autre annonce de lui-même — exactement la limite du dérouleur de la section 3, qui ne connaît de l'adversaire que la formule qu'on lui donne.

In [6]:
// --- Les trois bots, chacun declarant sa regle ---
// Un bot lit la DECLARATION de son adversaire (le jouet ne voit que le texte,
// pas la fonction : c'est la caricature assumee du paragraphe precedent).

/// nom -> (decision, declaration que les autres lisent)
type Bot = (&'static str, fn(&str) -> &'static str, &'static str);

/// Coopere toujours.
#[allow(dead_code)]
fn coopere_toujours(_decl: &str) -> &'static str { "C" }

/// Trahit toujours.
#[allow(dead_code)]
fn trahit_toujours(_decl: &str) -> &'static str { "D" }

/// Coopere si la declaration adverse garantit la cooperation contre
/// un cooperateur conditionnel -- c'est la clause de Payor, en texte.
#[allow(dead_code)]
fn fairbot_payorien(decl: &str) -> &'static str {
    if decl.contains("coopere toujours") || decl.contains("coopere si l'adversaire") { "C" } else { "D" }
}

/// Le gain du joueur `mien` quand l'autre joue `sien`.
#[allow(dead_code)]
fn gain(mien: &str, sien: &str) -> i32 {
    if mien == "C" && sien == "C" { return 3; }
    if mien == "C" { return 0; }
    if sien == "C" { 5 } else { 1 }
}

/// Les decisions des deux bots : chacun lit la declaration de l'autre.
#[allow(dead_code)]
fn jouer(bots: &[Bot], nom_a: &str, nom_b: &str) -> (String, String) {
    let ia = bots.iter().position(|b| b.0 == nom_a).expect("bot connu");
    let ib = bots.iter().position(|b| b.0 == nom_b).expect("bot connu");
    let decision_a = (bots[ia].1)(bots[ib].2);
    let decision_b = (bots[ib].1)(bots[ia].2);
    (decision_a.to_string(), decision_b.to_string())
}

let bots: Vec<Bot> = vec![
    ("cooperatebot", coopere_toujours, "coopere toujours"),
    ("defectbot", trahit_toujours, "trahit toujours"),
    ("fairbot_payorien", fairbot_payorien, "coopere si l'adversaire declare coopere"),
];

// --- Matrice des gains ---
let noms: Vec<&str> = bots.iter().map(|b| b.0).collect();
let mut entete = format!("{:<18}", "");
for n in &noms { entete.push_str(&format!("{:>18}", n)); }
println!("{}", entete);
println!("{}", "-".repeat(18 + 18 * noms.len()));
for a in &noms {
    let mut ligne = format!("{:<18}", a);
    for b in &noms {
        let (ca, cb) = jouer(&bots, a, b);
        ligne.push_str(&format!("{:>18}", format!("{}/{} {}", ca, cb, gain(&ca, &cb))));
    }
    println!("{}", ligne);
}

// --- Ce que la section 3 contraint, et ce qu'elle ne contraint pas ---
assert_eq!(jouer(&bots, "fairbot_payorien", "cooperatebot"), ("C".to_string(), "C".to_string()));     // ferme en 1 monde
assert_eq!(jouer(&bots, "fairbot_payorien", "fairbot_payorien"), ("C".to_string(), "C".to_string())); // ferme en 2 mondes

let (ca, cb) = jouer(&bots, "fairbot_payorien", "defectbot");
println!();
println!("P vs DefectBot (jouet) : {}/{} -> gain {}", ca, cb, gain(&ca, &cb));
println!("L'analyse de la section 3 rend 'sat' ici : le jouet choisit,");
println!("il ne derive pas. Les deux assertions ci-dessus, elles, sont contraintes.");

                        cooperatebot         defectbot  fairbot_payorien


------------------------------------------------------------------------


cooperatebot                   C/C 3             C/D 0             C/C 3


defectbot                      D/C 5             D/D 1             D/D 1


fairbot_payorien               C/C 3             D/D 1             C/C 3


P vs DefectBot (jouet) : D/D -> gain 1


L'analyse de la section 3 rend 'sat' ici : le jouet choisit,


il ne derive pas. Les deux assertions ci-dessus, elles, sont contraintes.


### Lecture du tournoi

Diagonale : `(3,3)`, `(1,1)`, `(3,3)`. Le FairBot payorien coopère contre le coopérateur inconditionnel et contre lui-même — les deux seules cases que la section 3 **contraint**, et les deux `assert` du code le vérifient.

La case qui instruit est `P vs DefectBot`. Le jouet rend `(1,1)`, et c'est exactement le piège : on serait tenté de lire ce `1/1` comme « le FairBot payorien trahit contre DefectBot, donc il est prudent ». La section 3 dit autre chose — `sat`, deux mondes — et la différence n'est pas une nuance :

- le jouet **applique une règle de correspondance textuelle** ; il n'a pas de notion de preuve, donc rien ne l'empêche de rendre `D` ;
- l'analyse de Kripke **laisse les deux issues ouvertes** : le modèle à deux mondes est un modèle *des prémisses*, il ne prouve pas que le bot joue `C` en vrai ;
- conclure « il trahit » demande une hypothèse supplémentaire, la cohérence du système de preuve du bot. Le post la pose explicitement ; beaucoup de relectures l'oublient.

Autrement dit : le tournoi et la réfutation ne sont pas deux mesures de la même chose. Le tournoi **exhibe** un comportement ; la réfutation **borne** ce qui est forcé. Confondre les deux, c'est prendre un modèle pour un théorème.

## 5. Deux mondes au lieu de quatre

Le titre du post est une comparaison : la clause de Payor réfute en **deux** mondes là où la route löbienne en demande **quatre**. Il faut séparer ce que ce carnet a mesuré de ce qu'il ne fait que citer, parce que la moitié citée est la moitié qui compte pour le titre.

| Ce qui est en jeu | Statut ici | Détail |
|---|---|---|
| Réfutation de `¬P` sous la clause de Payor, contre un Payorien | **mesuré** — 1 monde contre CooperateBot, 2 contre un Payorien | sections 3 et 4, `assert` du carnet |
| Réfutation de `¬P` contre DefectBot | **mesuré** — `sat`, modèle à 2 mondes | frontière de l'hypothèse de cohérence |
| Route löbienne : « quatre mondes » | **cité**, non re-mesuré | le dérouleur rend `incomplet` sur la phrase de Henkin seule |

Pourquoi le « quatre » ne se re-mesure pas avec cet instrument. La route löbienne part de `F(X) ↔ □(X(F))`, donc pour `F` contre `F` il faut établir un point fixe : `F(F)=C` ssi `□(F(F)=C)`. La preuve du post introduit une **phrase de Henkin** `H ↔ □H` et une **auxiliaire** `L ↔ (□L → H)` ; c'est cette auxiliaire qui rend la dérivation finie, et elle la rend finie **parce qu'elle est une instance de l'axiome de Löb** : `□(□L→H) → L`-style, appliqué à un antécédent qu'on vient de dériver. L'axiome de Löb encode la bonne-fondation des cadres de GL.

Notre dérouleur n'a ni l'axiome ni la bonne-fondation : il empile des contraintes de boîte et ouvre un monde de plus à chaque fois qu'une obligation reste ouverte. Sur `H ↔ □H` avec `¬H`, il en ouvre indéfiniment — d'où l'`incomplet` de la section 3. **Ce n'est pas un échec de la réfutation, c'est la limite de l'instrument**, et la dire fait partie du résultat.

Ce que la comparaison garde de solide, et qui est le vrai contenu pédagogique : les deux routes ne coûtent pas la même chose parce qu'elles ne prouvent pas la même chose. La clause de Payor ajoute une **boîte en position d'antécédent** — `□P` sous le `□` extérieur — et cette boîte intérieure est ce qui referme la réfutation en descendant une seule fois. Lire `□(□P→Q)` comme « je coopère si je peux prouver que ma coopération implique la tienne » est exactement ce qui rend la preuve courte : le bot se demande ce qu'il peut prouver **de sa propre coopération**, pas seulement celle de l'adversaire.

## 6. Les variantes du fil — citées, pas re-mesurées

La discussion qui suit le post propose trois bots de plus. Aucun n'est implémenté ici, et c'est un choix : leurs définitions sortent du fragment que le dérouleur traite (voir l'avertissement en fin de section). Les définitions sont citées telles quelles.

**PrudentBot d'origine** — `P(X) ↔ □(X(P) ∧ ◊¬X(DB))` : il coopère s'il prouve que l'adversaire coopère avec lui **et** qu'il existe un monde où cet adversaire trahit contre DefectBot. L'auteur du post juge lui-même cette référence à DefectBot « un peu bricolée » (*a little hacked*), et c'est ce qui motive la suite.

**SharkBot** — `S(X) ↔ □(□S(X) → X(S)) ∧ ◊(X(S) → ◊S(X))` : la clause de Payor, plus une seconde condition sur les diamants. L'auteur annonce « je suis assez sûr » (*I'm pretty sure*) qu'il coopère avec lui-même et avec FairBot, et qu'il trahit contre CooperateBot. **« Assez sûr » n'est pas un théorème** : c'est une conjecture de discussion, et ce carnet ne la vérifie pas.

**PrudentBot payorien** — `P ↔ □(□P→Q) ∧ □(□¬P→¬Q)`, obtenu en simulant l'adversaire deux fois : une fois sous l'hypothèse d'une preuve que je coopère, une fois sous l'hypothèse d'une preuve que je défais. La réfutation part de `¬P` et rend `◊(□P∧¬Q) ∨ ◊(□¬P∧Q)` : **deux** jeux de contraintes à réfuter, un par simulation. L'auteur conclut lui-même : « je ne suis pas vraiment sûr de comment tout cela va tourner » (*I'm not really sure how any of this will turn out*).

### Pourquoi ces variantes ne passent pas dans le dérouleur

`◊(□P∧¬Q) ∨ ◊(□¬P∧Q)` contient une **disjonction**, et `¬P` sur `P ≡ A∧B` produit `¬A ∨ ¬B`. Or le dérouleur de la section 3 ne décompose **ni les disjonctions ni les négations de conjonction** : il ne connaît que `¬¬`, `∧`, `→`, `¬(A→B)`, `□`, `¬◊`, `¬□` et `◊`. Une formule qu'il n'a pas décomposée reste dans le monde sans être traitée, et le dérouleur rendrait `sat` — **un `sat` qui ne voudrait rien dire**. C'est la raison pour laquelle les quatre cas de la section 3 ont été choisis sans disjonction : sur ce fragment-là, le résultat est fiable ; en dehors, il ne l'est pas.

Cette limite n'est pas propre au code. James Payor note dans le fil que la traduction `◊ = ¬□¬` **ne reste pas fidèle à l'intention** (*the "not box not" translation doesn't actually stay very faithful to the intent*), et que la manipulation des longueurs de preuve y est délicate. Le dérouleur implémente exactement cette traduction-là : quand il rencontre `¬□φ`, il ouvre un monde avec `¬φ`. C'est commode, c'est mesurable, et ce n'est pas la même chose que « il existe un monde où φ est vrai » au sens de la logique de la prouvabilité.

## 7. Exercices

Trois exercices, du plus mécanique au plus conceptuel. Les cellules sont des squelettes : elles s'exécutent telles quelles et rendent `None` tant qu'elles ne sont pas complétées. Aucune ne lève d'erreur volontairement.

**Exercice 1 — réfuter avec des disjonctions.** Le dérouleur ne décompose pas `¬(A∧B)`, qui vaut `¬A ∨ ¬B`. Écrire un `refute_branche` qui, en présence d'une disjonction à satisfaire, **explore les deux branches** et ne déclare `unsat` que si toutes les branches se ferment. Le tester sur les quatre cas de la section 3 : il doit rendre exactement les mêmes statuts.

In [7]:
// Exercice 1 — refutation avec disjonctions
/// AST des formules modales : la recursion est portee par `Box`.
/// `Clone + PartialEq + Eq + Hash` reproduit la dataclass `frozen` Python :
/// l'egalite et le hachage sont structurels.
#[derive(Clone, PartialEq, Eq, Hash)]
#[allow(dead_code)]
enum Formule {
    Atome(String),
    Non(Box<Formule>),
    Imp(Box<Formule>, Box<Formule>),
    Et(Box<Formule>, Box<Formule>),
    Boite(Box<Formule>),
    Diamant(Box<Formule>),
}

/// Comme refute, mais decompose les disjonctions en explorant les deux branches.
///
/// Retour attendu :
///   Some("unsat")     si TOUTES les branches se ferment
///   Some("sat")       si une branche sature sans contradiction
///   Some("incomplet") si une branche atteint la borne
///   None              tant que l'exercice n'est pas complete
#[allow(dead_code)]
fn refute_branche(_premisses: &[Formule], _max_profondeur: usize) -> Option<String> {
    let resultat: Option<String> = None; // TODO etudiant : explorer les deux branches, n'accepter
    //                  unsat que si toutes se ferment
    resultat
}

println!("Exercice 1 — refute_branche reste a completer.");
println!("Controle attendu une fois complete : les quatre cas de la section 3");
println!("doivent rendre exactement les memes (statut, borne) que le tableau.");

Exercice 1 — refute_branche reste a completer.


Controle attendu une fois complete : les quatre cas de la section 3


doivent rendre exactement les memes (statut, borne) que le tableau.


**Exercice 2 — le PrudentBot payorien.** Construire `COND_PRUDENT` à partir de la définition citée en section 6, réfuter `¬P` avec le dérouleur **sans** branchement, puis avec `refute_branche`. Expliquer en trois phrases pourquoi le premier résultat ne répond pas à la question.

In [8]:
// Exercice 2 — le PrudentBot payorien
// Definition citee en section 6 :   P <-> []([]P -> Q) /\ []([]~P -> ~Q)
/// AST des formules modales : la recursion est portee par `Box`.
/// `Clone + PartialEq + Eq + Hash` reproduit la dataclass `frozen` Python :
/// l'egalite et le hachage sont structurels.
#[derive(Clone, PartialEq, Eq, Hash)]
#[allow(dead_code)]
enum Formule {
    Atome(String),
    Non(Box<Formule>),
    Imp(Box<Formule>, Box<Formule>),
    Et(Box<Formule>, Box<Formule>),
    Boite(Box<Formule>),
    Diamant(Box<Formule>),
}

// Puis, pour le meme jeu [COND_PRUDENT, non(P_AT)] :
//   statut_sans  = refute(...)            -> que rend le derouleur, et pourquoi ce n'est pas une reponse ?
//   statut_avec  = refute_branche(...)    -> ce que donne l'exploration des deux branches
#[allow(unused_variables)]
let cond_prudent: Option<Formule> = None; // TODO etudiant : construire la formule
#[allow(unused_variables)]
let resultat_ex2: Option<String> = None;   // TODO etudiant

println!("Exercice 2 — COND_PRUDENT et les deux refutations restent a completer.");
println!("Rappel : la negation de ~P developpe deux branches, []P et ~Q d'une part,");
println!("         []~P et Q de l'autre.");

Exercice 2 — COND_PRUDENT et les deux refutations restent a completer.


Rappel : la negation de ~P developpe deux branches, []P et ~Q d'une part,


         []~P et Q de l'autre.


**Exercice 3 — compter les mondes.** Sur le cas löbien, relever le nombre de mondes créés pour chaque borne de 1 à 6. En déduire pourquoi aucune borne ne rendra ce cas `sat` ni `unsat`, et ce qu'il faudrait ajouter au dérouleur pour fermer la réfutation.

In [9]:
// Exercice 3 — compter les mondes crees sur le cas lobien
/// AST des formules modales : la recursion est portee par `Box`.
/// `Clone + PartialEq + Eq + Hash` reproduit la dataclass `frozen` Python :
/// l'egalite et le hachage sont structurels.
#[derive(Clone, PartialEq, Eq, Hash)]
#[allow(dead_code)]
enum Formule {
    Atome(String),
    Non(Box<Formule>),
    Imp(Box<Formule>, Box<Formule>),
    Et(Box<Formule>, Box<Formule>),
    Boite(Box<Formule>),
    Diamant(Box<Formule>),
}

/// Rend le nombre de mondes existants au moment ou le derouleur s'arrete.
#[allow(dead_code)]
fn compte_mondes(_premisses: &[Formule], _borne: usize) -> Option<usize> {
    None // TODO etudiant
}

// Releve attendu : un dictionnaire {borne: nombre_de_mondes} pour borne de 1 a 6,
// obtenu sur CAS["Lob : H <-> []H"]. Conclusion attendue en deux phrases :
// pourquoi aucune borne ne rendra ce cas sat ni unsat, et ce qu'il faudrait
// ajouter au derouleur pour fermer la refutation.
#[allow(unused_variables)]
let releve: Option<Vec<(usize, usize)>> = None; // TODO etudiant

println!("Exercice 3 — compte_mondes et le releve restent a completer.");

Exercice 3 — compte_mondes et le releve restent a completer.


## À retenir

- La clause de Payor est `P ↔ □(□P→Q)` : le bot se demande ce qu'il peut prouver **de sa propre coopération**, pas seulement de celle de l'adversaire. C'est la boîte intérieure qui fait la différence avec la clause löbienne `F(X) ↔ □X(F)`.
- Sur un cadre de Kripke, cette clause se réfute en **deux** mondes contre un autre Payorien, et en **un** contre CooperateBot. Les deux profondeurs sont mesurées et vérifiées par `assert`.
- Contre DefectBot, la réfutation **ne conclut pas** : elle rend un modèle à deux mondes. Ce `sat` est la frontière de l'hypothèse de cohérence, pas une preuve que le bot trahit.
- La route löbienne demande, dans le post, quatre mondes et une auxiliaire `L ↔ (□L→H)`. Ce carnet la **cite** sans la re-mesurer : le dérouleur n'a ni l'axiome de Löb ni la bonne-fondation, et rend `incomplet` sur `H ↔ □H`. Un `incomplet` est un résultat honnête ; le maquiller en `sat` ou en `unsat` ne le serait pas.
- Le tournoi-jouet illustre sans démontrer : il applique une correspondance textuelle là où la logique borne ce qui est **forcé**. Les deux `assert` du tournoi couvrent les seules cases contraintes.

## Vérifiez votre compréhension

1. Pourquoi `□φ` est-il vrai sur un monde sans successeur ? Quelle est la conséquence pour le modèle `sat` de `P vs DefectBot` ?
2. Dans la réfutation `P vs Payorian Q`, à quel monde la contradiction apparaît-elle, et quelles deux formules s'y rencontrent ?
3. Le dérouleur rend `incomplet` sur le cas löbien. Est-ce parce que les prémisses sont cohérentes, contradictoires, ou parce que l'instrument ne sait pas conclure ?
4. Pourquoi un `sat` obtenu sur une formule contenant une disjonction non décomposée ne vaut-il rien ?

<details>
<summary>Éléments de réponse</summary>

1. La clause de forcing de `□` est un « pour tout » sur les successeurs : sur un monde qui n'en a aucun, l'ensemble vide est universellement vrai. C'est ce qui rend le modèle `sat` de `P vs DefectBot` cohérent : le monde 1 est terminal, donc `□P` y est vacuement vrai ; mais c'est aussi ce qui limite sa portée — un modèle du cadre employé, pas une preuve sur le bot.
2. La contradiction apparaît au monde 2 : `¬P` — posé par la négation de la définition de Q — et `P` — posé par le `□P` du monde 1 — s'y rencontrent. Les deux définitions sont en jeu, chacune atteinte par un chemin différent : c'est le « deux mondes au lieu de quatre » du titre du post.
3. Parce que l'instrument ne sait pas conclure. Sur `H ↔ □H` avec `¬H`, le dérouleur ouvre un monde de plus à chaque passe ; faute de l'axiome de Löb et de la bonne-fondation des cadres de GL, il bute sur sa borne et rend `incomplet` — ni cohérent, ni contradictoire.
4. Parce que le dérouleur ne décompose ni les disjonctions ni les négations de conjonction : une formule qu'il ne sait pas traiter reste dans le monde sans être traitée, et la saturation s'arrête en rendant `sat`. Les prémisses n'ont donc pas été satisfaites : ce `sat` ne prouve rien.

</details>

## Pour aller plus loin

- Le post d'origine et sa discussion : *Payorian cooperation is easy with Kripke frames*, Alignment Forum — archive du gisement, `GameTheory/Cooperation, Institutions and LLM Agents/2026-03-09 - transhumanist_atom_understander - Payorian cooperation is easy with Kripke frames.pdf`.
- Dans ce dépôt : `Dynamics-04b-FairBot-Loeb-Lean` pour la route löbienne et sa mécanisation partielle, `Dynamics-05-Agents-Bornes-Python` pour les agents bornés, `Dynamics-06-Preuves-Bornees-Cout-du-Raisonnement-Python` pour le coût du raisonnement et l'Open Problem 3.
- Le moteur de Kripke de la section 2 est une copie pédagogique déclarée de `SymbolicAI/Tweety/Tweety-03b-Modal-Lab-Lean-Python` : le lire pour la logique modale générale dont ce carnet n'utilise qu'un fragment.